In [2]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.spatial.distance import cdist 

# Make sure GOC is bi-directional.

In [17]:
humanMouseRecipricol = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/humanMouseReciprical.txt", sep="\t").rename(columns={"Gene stable ID": "MouseID", "Human gene stable ID": "HumanID"})

In [18]:
orthologTable = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupeBioType.parquet")

In [22]:
orthologTableHumanGOC = orthologTable.merge(humanMouseRecipricol, left_on=["Gene stable ID", "Mouse gene stable ID"], right_on=["HumanID", "MouseID"], how="left").drop(columns=["MouseID", "HumanID"])
orthologTableHumanGOC.insert(3, "Human Gene-order conservation score", orthologTableHumanGOC.pop("Human Gene-order conservation score"))

# Identifying Duplicated Species

In [72]:
orthoDist = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDist.parquet")
display(orthoDist)

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score,EuclidDist,EuclidDistNorm,EuclidDistLog,PearDist,TEC
0,ENSG00000198888,ENSMUSG00000064341,ortholog_one2one,50.0,44492.547302,0.435805,6.007114,0.209431,0.0000
1,ENSG00000198763,ENSMUSG00000064345,ortholog_one2one,75.0,52572.771576,0.509245,7.318626,0.297247,0.0000
2,ENSG00000198804,ENSMUSG00000064351,ortholog_one2one,100.0,82619.259132,0.416690,6.678323,0.235434,0.0000
3,ENSG00000198712,ENSMUSG00000064354,ortholog_one2one,100.0,146743.617287,NaN,40.896612,NaN,NaN
4,ENSG00000228253,ENSMUSG00000064356,ortholog_one2one,100.0,72933.323800,NaN,38.358476,NaN,NaN
...,...,...,...,...,...,...,...,...,...
28009,ENSG00000081692,ENSMUSG00000036819,ortholog_one2one,75.0,16.599178,0.668685,3.412393,0.996874,0.0625
28010,ENSG00000157873,ENSMUSG00000022074,ortholog_one2many,0.0,194.970650,0.543398,11.676671,0.384453,0.2500
28011,ENSG00000157873,ENSMUSG00000042333,ortholog_one2many,100.0,178.921829,0.834154,9.002332,0.688115,0.0625
28012,ENSG00000132676,ENSMUSG00000068921,ortholog_one2one,75.0,243.094613,0.516088,4.801605,0.922319,0.0000


In [73]:
# Identifies the number of mouse genes associated with each human gene.
groupedHumanGene = orthoDist.groupby("Gene stable ID")["Mouse gene stable ID"].apply(list).reset_index(name="Mouse gene stable ID")

# Adds in the homology type to the dataframe.
groupedHumanGene = groupedHumanGene.merge(orthoDist.loc[:, ["Gene stable ID", "Mouse homology type"]])

# Identifies the human genes that have more than one mouse gene associated with them.
groupedHumanGene["Num Mouse Dupes"] = groupedHumanGene["Mouse gene stable ID"].apply(len)

# Creates a new column that identifies which one-to-many gene experienced a duplication event in humans. 
groupedHumanGene["Duplicated Species"] = np.where((groupedHumanGene["Gene stable ID"].isin(groupedHumanGene[groupedHumanGene["Num Mouse Dupes"] > 1]["Gene stable ID"])) & (groupedHumanGene["Mouse homology type"] == "ortholog_one2many"), "Mouse", "NA")

In [74]:
groupedHumanGene

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Num Mouse Dupes,Duplicated Species
0,ENSG00000000003,[ENSMUSG00000067377],ortholog_one2one,1,NA
1,ENSG00000000005,[ENSMUSG00000031250],ortholog_one2one,1,NA
2,ENSG00000000419,[ENSMUSG00000078919],ortholog_one2one,1,NA
3,ENSG00000000457,[ENSMUSG00000026584],ortholog_one2one,1,NA
4,ENSG00000000460,[ENSMUSG00000041406],ortholog_one2one,1,NA
...,...,...,...,...,...
28009,ENSG00000310576,[ENSMUSG00000035595],ortholog_one2one,1,NA
28010,ENSG00000310579,[nan],NaN,1,NA
28011,ENSG00000310583,[nan],NaN,1,NA
28012,ENSG00000310590,[nan],NaN,1,NA


In [75]:
# Transfers the information above to the orthologTableDist dataframe.
orthoDistHumanDup = orthoDist.merge(groupedHumanGene.loc[:, ["Gene stable ID", "Duplicated Species", "Num Mouse Dupes"]], left_on="Gene stable ID", right_on="Gene stable ID", how="outer")

In [76]:
# Same steps as three cells above, but for mouse genes.
groupedMouseGene = orthoDist.groupby("Mouse gene stable ID")["Gene stable ID"].apply(list).reset_index(name="Gene stable ID")
groupedMouseGene = groupedMouseGene.merge(orthoDist.loc[:, ["Mouse gene stable ID", "Mouse homology type"]])
groupedMouseGene["Num Human Dupes"] = groupedMouseGene["Gene stable ID"].apply(len)
groupedMouseGene["Duplicated Species"] = np.where((groupedMouseGene["Mouse gene stable ID"].isin(groupedMouseGene[groupedMouseGene["Num Human Dupes"] > 1]["Mouse gene stable ID"])) & (groupedMouseGene["Mouse homology type"] == "ortholog_one2many"), "Human", "NA")

In [77]:
groupedMouseGene[groupedMouseGene["Mouse gene stable ID"] == "ENSMUSG00000030945"]

,Mouse gene stable ID,Gene stable ID,Mouse homology type,Num Human Dupes,Duplicated Species
8043,ENSMUSG00000030945,"[ENSG00000066813, ENSG00000183747]",ortholog_many2many,2,NA
8044,ENSMUSG00000030945,"[ENSG00000066813, ENSG00000183747]",ortholog_many2many,2,NA


In [78]:
groupedMouseGene

,Mouse gene stable ID,Gene stable ID,Mouse homology type,Num Human Dupes,Duplicated Species
0,ENSMUSG00000000001,[ENSG00000065135],ortholog_one2one,1,NA
1,ENSMUSG00000000028,[ENSG00000093009],ortholog_one2one,1,NA
2,ENSMUSG00000000037,[ENSG00000102098],ortholog_one2one,1,NA
3,ENSMUSG00000000049,[ENSG00000091583],ortholog_one2one,1,NA
4,ENSMUSG00000000056,[ENSG00000141562],ortholog_one2one,1,NA
...,...,...,...,...,...
22464,ENSMUSG00000144248,[ENSG00000289360],ortholog_one2one,1,NA
22465,ENSMUSG00000144259,[ENSG00000288706],ortholog_one2one,1,NA
22466,ENSMUSG00000144287,[ENSG00000255154],ortholog_one2one,1,NA
22467,ENSMUSG00001074846,[ENSG00000229972],ortholog_one2one,1,NA


In [79]:
# Transfers the information above to the orthologDistTable dataframe.
orthoDistHumanMouseDup = orthoDistHumanDup.merge(groupedMouseGene.loc[:, ["Mouse gene stable ID", "Duplicated Species", "Num Human Dupes"]], on="Mouse gene stable ID", how="outer")

# Because there are two separate "Duplicated Species" columns, I am going to combine their information into one.
# If the human "Duplicated Species" column is "NA" then we use the mouse "Duplicated Species" column. It will either be "Mouse" or stay "NA."
orthoDistHumanMouseDup["Duplicated Species"] = np.where(orthoDistHumanMouseDup["Duplicated Species_x"] == "NA", orthoDistHumanMouseDup["Duplicated Species_y"], orthoDistHumanMouseDup["Duplicated Species_x"])

# Drop the two separate "Duplicated Species" columns.
orthoDistHumanMouseDup = orthoDistHumanMouseDup.loc[:, ~orthoDistHumanMouseDup.columns.isin(["Duplicated Species_x", "Duplicated Species_y"])].drop_duplicates()
display(orthoDistHumanMouseDup)

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score,EuclidDist,EuclidDistNorm,EuclidDistLog,PearDist,TEC,Num Mouse Dupes,Num Human Dupes,Duplicated Species
0,ENSG00000065135,ENSMUSG00000000001,ortholog_one2one,100.0,197.746445,0.278561,6.701999,0.151442,0.000000,1,1.0,NA
1,ENSG00000093009,ENSMUSG00000000028,ortholog_one2one,100.0,24.533616,1.024816,5.058507,1.001638,0.250000,1,1.0,NA
2,ENSG00000102098,ENSMUSG00000000037,ortholog_one2one,100.0,2.084639,0.471628,1.444542,0.161786,0.166667,1,1.0,NA
3,ENSG00000091583,ENSMUSG00000000049,ortholog_one2one,100.0,1776.546401,0.002316,2.516381,0.000003,0.500000,1,1.0,NA
4,ENSG00000141562,ENSMUSG00000000056,ortholog_one2one,100.0,70.950296,0.525139,3.511421,0.661562,0.000000,1,1.0,NA
...,...,...,...,...,...,...,...,...,...,...,...,...
3884923,ENSG00000310562,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN
3884924,ENSG00000310579,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN
3884925,ENSG00000310583,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN
3884926,ENSG00000310590,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN


In [80]:
orthoDistHumanMouseDup.columns[-1:]

Index(['Duplicated Species'], dtype='str')

In [81]:
newColOrder = list(orthoDistHumanMouseDup.columns[:-3]) + list(orthoDistHumanMouseDup.columns[-2:-1]) + list(orthoDistHumanMouseDup.columns[-3:-2]) + list(orthoDistHumanMouseDup.columns[-1:])
orthoDistHumanMouseDup = orthoDistHumanMouseDup.loc[:, newColOrder]

In [82]:
orthoDistHumanMouseDup.to_csv("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupe.csv", index=False)
orthoDistHumanMouseDup.to_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupe.parquet", index=False)

In [83]:
gtexEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/gtexExpressionProfile.parquet")
emtabEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/emtabExpressionProfile.parquet")

In [84]:
gtexEP

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSG00000000003,5.799691,22.906008,18.107586,3.414238,15.745596,23.313591,7.128223,10.594314,protein_coding
ENSG00000000005,0.169146,0.753748,0.284215,0.258039,0.956031,0.018972,0.050229,0.206583,protein_coding
ENSG00000000419,21.387617,40.514694,43.498722,24.609381,25.236029,22.603436,22.147223,37.476429,protein_coding
ENSG00000000457,2.830432,6.585192,6.015115,1.902403,3.748399,4.093346,3.044870,5.295540,protein_coding
ENSG00000000460,1.347589,2.231597,2.198975,0.689768,0.988887,1.240150,0.699461,1.596039,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSG00000310553,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310554,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310555,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA


In [85]:
emtabEP

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSMUSG00000000001,34.911248,129.180584,143.270691,72.898048,72.920467,45.744554,8.901608,54.003405,protein_coding
ENSMUSG00000000003,0.000000,0.000000,0.000000,0.053683,0.000000,0.000000,0.000000,0.000000,protein_coding
ENSMUSG00000000028,2.236124,11.323525,5.613315,22.680435,2.280480,0.871723,0.181636,3.491172,protein_coding
ENSMUSG00000000031,2.567429,1.137826,2650.257043,17.372786,0.758899,1.307234,1.672216,2.287438,lncRNA
ENSMUSG00000000037,1.226929,2.675817,3.285196,0.774847,0.333726,0.011223,0.000000,0.287549,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSMUSG00000109574,0.332800,0.047076,0.021773,0.015338,0.008382,0.000000,0.000000,0.000000,TEC
ENSMUSG00000109575,1.098742,0.000000,0.000000,0.009073,0.000000,0.000000,0.000000,0.000000,TEC
ENSMUSG00000109576,0.022506,0.000000,0.000000,0.024401,0.012853,0.000000,0.000000,0.000000,TEC


In [86]:
orthoDistHumanMouseDupBioType = orthoDistHumanMouseDup.merge(gtexEP.loc[:, ["Gene type"]], left_on="Gene stable ID", right_index=True, how="left")
orthoDistHumanMouseDupBioType = orthoDistHumanMouseDupBioType.rename(columns={"Gene type": "Human Gene Type"})

In [87]:
orthoDistHumanMouseDupBioType = orthoDistHumanMouseDupBioType.merge(emtabEP.loc[:, ["Gene type"]], left_on="Mouse gene stable ID", right_index=True, how="left")
orthoDistHumanMouseDupBioType = orthoDistHumanMouseDupBioType.rename(columns={"Gene type": "Mouse Gene Type"})

In [88]:
orthoDistHumanMouseDupBioType

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score,EuclidDist,EuclidDistNorm,EuclidDistLog,PearDist,TEC,Num Human Dupes,Num Mouse Dupes,Duplicated Species,Human Gene Type,Mouse Gene Type
0,ENSG00000065135,ENSMUSG00000000001,ortholog_one2one,100.0,197.746445,0.278561,6.701999,0.151442,0.000000,1.0,1,NA,protein_coding,protein_coding
1,ENSG00000093009,ENSMUSG00000000028,ortholog_one2one,100.0,24.533616,1.024816,5.058507,1.001638,0.250000,1.0,1,NA,protein_coding,protein_coding
2,ENSG00000102098,ENSMUSG00000000037,ortholog_one2one,100.0,2.084639,0.471628,1.444542,0.161786,0.166667,1.0,1,NA,protein_coding,protein_coding
3,ENSG00000091583,ENSMUSG00000000049,ortholog_one2one,100.0,1776.546401,0.002316,2.516381,0.000003,0.500000,1.0,1,NA,protein_coding,protein_coding
4,ENSG00000141562,ENSMUSG00000000056,ortholog_one2one,100.0,70.950296,0.525139,3.511421,0.661562,0.000000,1.0,1,NA,protein_coding,protein_coding
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3884923,ENSG00000310562,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN
3884924,ENSG00000310579,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN
3884925,ENSG00000310583,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN
3884926,ENSG00000310590,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN


In [89]:
orthoDistHumanMouseDupBioType.to_csv("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupeBioType.csv")
orthoDistHumanMouseDupBioType.to_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupeBioType.parquet")

# Statistical Analysis

In [3]:
data = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupeBioType.parquet")
dataProteinOnly = data[(data["Human Gene Type"] == "protein_coding") & (data["Mouse Gene Type"] == "protein_coding")]

In [30]:
dataProteinOnly

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score,EuclidDist,EuclidDistNorm,EuclidDistLog,PearDist,TEC,Num Human Dupes,Num Mouse Dupes,Duplicated Species,Human Gene Type,Mouse Gene Type
0,ENSG00000065135,ENSMUSG00000000001,ortholog_one2one,100.0,197.746445,0.278561,6.701999,0.151442,0.000000,1.0,1,NA,protein_coding,protein_coding
1,ENSG00000093009,ENSMUSG00000000028,ortholog_one2one,100.0,24.533616,1.024816,5.058507,1.001638,0.250000,1.0,1,NA,protein_coding,protein_coding
2,ENSG00000102098,ENSMUSG00000000037,ortholog_one2one,100.0,2.084639,0.471628,1.444542,0.161786,0.166667,1.0,1,NA,protein_coding,protein_coding
3,ENSG00000091583,ENSMUSG00000000049,ortholog_one2one,100.0,1776.546401,0.002316,2.516381,0.000003,0.500000,1.0,1,NA,protein_coding,protein_coding
4,ENSG00000141562,ENSMUSG00000000056,ortholog_one2one,100.0,70.950296,0.525139,3.511421,0.661562,0.000000,1.0,1,NA,protein_coding,protein_coding
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3855627,ENSG00000182854,ENSMUSG00000109528,ortholog_one2one,50.0,0.007129,NaN,0.009994,NaN,NaN,1.0,1,NA,protein_coding,protein_coding
3855628,ENSG00000183303,ENSMUSG00000109542,ortholog_one2many,75.0,0.025535,NaN,0.020456,NaN,NaN,1.0,21,Mouse,protein_coding,protein_coding
3855649,ENSG00000145700,ENSMUSG00000109561,ortholog_one2one,100.0,0.362324,1.136140,0.434832,1.194783,NaN,1.0,1,NA,protein_coding,protein_coding
3855650,ENSG00000181143,ENSMUSG00000109564,ortholog_one2one,75.0,11.742337,0.616651,3.766970,0.237058,NaN,1.0,1,NA,protein_coding,protein_coding


In [10]:
dataProteinOnly[dataProteinOnly["Mouse homology type"] == "ortholog_one2one"]["EuclidDistNorm"].dropna()

0          0.278561
1          1.024816
2          0.471628
3          0.002316
4          0.525139
             ...   
3855599    0.847650
3855623    0.600777
3855649    1.136140
3855650    0.616651
3855651    0.652040
Name: EuclidDistNorm, Length: 15889, dtype: float64

In [13]:
dataProteinOnly[dataProteinOnly["Mouse homology type"] == "ortholog_one2one"]["TEC"].dropna()

0          0.000000
1          0.250000
2          0.166667
3          0.500000
4          0.000000
             ...   
3855538    0.166667
3855539    0.000000
3855541    0.000000
3855599    0.416667
3855623    0.000000
Name: TEC, Length: 14186, dtype: float64

In [31]:
statsDF = pd.DataFrame({
    "Category": ["one-to-one", 
                 "one-to-many", "one-to-many, duplicated in human", "one-to-many, duplicated in human, GOC=0", "one-to-many, duplicated in human, GOC=25", "one-to-many, duplicated in human, GOC=50", "one-to-many, duplicated in human, GOC=75", "one-to-many, duplicated in human, GOC=100",
                 "one-to-many, duplicated in mouse", "one-to-many, duplicated in mouse, GOC=0", "one-to-many, duplicated in mouse, GOC=25", "one-to-many, duplicated in mouse, GOC=50", "one-to-many, duplicated in mouse, GOC=75", "one-to-many, duplicated in mouse, GOC=100",
                 "many-to-many"],
    "n": "",
    "Median": "",
    "Mean": ""
})

In [32]:
statsDF

,Category,n,Median,Mean
0,one-to-one,,,
1,one-to-many,,,
2,"one-to-many, duplicated in human",,,
3,"one-to-many, duplicated in human, GOC=0",,,
4,"one-to-many, duplicated in human, GOC=25",,,
5,"one-to-many, duplicated in human, GOC=50",,,
6,"one-to-many, duplicated in human, GOC=75",,,
7,"one-to-many, duplicated in human, GOC=100",,,
8,"one-to-many, duplicated in mouse",,,
9,"one-to-many, duplicated in mouse, GOC=0",,,


In [33]:
translationTable = {
    "one-to-one": "ortholog_one2one",
    "one-to-many": "ortholog_one2many",
    "duplicated in human": "Human",
    "duplicated in mouse": "Mouse",
    "GOC=0": 0,
    "GOC=25": 25,
    "GOC=50": 50,
    "GOC=75": 75,
    "GOC=100": 100,
    "many-to-many": "ortholog_many2many"
}

In [34]:
statsDFArr = []
for distMetric in dataProteinOnly.columns[4:9]:
    nArr = []
    medianArr = []
    meanArr = []

    for category in statsDF["Category"]:
        categoryParts = category.split(", ")
        translatedParts = list(map(translationTable.get, categoryParts))
        if len(translatedParts) == 1:
            filteredDF = dataProteinOnly[dataProteinOnly["Mouse homology type"] == translatedParts[0]][distMetric].dropna()
        elif len(translatedParts) == 2:
            filteredDF = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedParts[0]) & (dataProteinOnly["Duplicated Species"] == translatedParts[1])][distMetric].dropna()
        else:
            filteredDF = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedParts[0]) & (dataProteinOnly["Duplicated Species"] == translatedParts[1]) & (dataProteinOnly["Mouse Gene-order conservation score"] == translatedParts[2])][distMetric].dropna()

        nArr.append(filteredDF.shape[0])
        medianArr.append(filteredDF.median())
        meanArr.append(filteredDF.mean())

    statsDF["n"] = nArr
    statsDF["Median"] = medianArr
    statsDF["Mean"] = meanArr
    statsDFArr.append(statsDF.copy())

In [35]:
statsDFArr

[                                     Category      n     Median         Mean
 0                                  one-to-one  16116  45.779092   255.650467
 1                                 one-to-many   1886   7.789615   461.516873
 2            one-to-many, duplicated in human    480  25.503485  1012.029178
 3     one-to-many, duplicated in human, GOC=0    221  29.224520   393.903635
 4    one-to-many, duplicated in human, GOC=25     46   4.922673    82.496779
 5    one-to-many, duplicated in human, GOC=50     69  17.227953    72.087825
 6    one-to-many, duplicated in human, GOC=75     28  23.071310   171.103285
 7   one-to-many, duplicated in human, GOC=100    116  49.904925  3320.357744
 8            one-to-many, duplicated in mouse   1405   5.020403   273.327189
 9     one-to-many, duplicated in mouse, GOC=0    567   1.765446   142.361515
 10   one-to-many, duplicated in mouse, GOC=25    143   0.118407   692.697554
 11   one-to-many, duplicated in mouse, GOC=50    198   2.765075

In [36]:
pValueDF = pd.DataFrame({
    "Group 1": ["one-to-one", "one-to-one",
                "one-to-many, duplicated in human",
                "many-to-many", "many-to-many",
                "one-to-many, duplicated in human, GOC=0", "one-to-many, duplicated in human, GOC=25", "one-to-many, duplicated in human, GOC=50", "one-to-many, duplicated in human, GOC=75", "one-to-many, duplicated in human, GOC=0",
                "one-to-many, duplicated in mouse, GOC=0", "one-to-many, duplicated in mouse, GOC=25", "one-to-many, duplicated in mouse, GOC=50", "one-to-many, duplicated in mouse, GOC=75", "one-to-many, duplicated in mouse, GOC=0"],

    "Group 2": ["one-to-many, duplicated in human", "one-to-many, duplicated in mouse",
                "one-to-many, duplicated in mouse", 
                "one-to-many, duplicated in human", "one-to-many, duplicated in mouse", 
                "one-to-many, duplicated in human, GOC=25", "one-to-many, duplicated in human, GOC=50", "one-to-many, duplicated in human, GOC=75", "one-to-many, duplicated in human, GOC=100", "one-to-many, duplicated in human, GOC=100",
                "one-to-many, duplicated in mouse, GOC=25", "one-to-many, duplicated in mouse, GOC=50", "one-to-many, duplicated in mouse, GOC=75", "one-to-many, duplicated in mouse, GOC=100", "one-to-many, duplicated in mouse, GOC=100"],
    "P-value": ""
})

In [37]:
pValueDF

,Group 1,Group 2,P-value
0,one-to-one,"one-to-many, duplicated in human",
1,one-to-one,"one-to-many, duplicated in mouse",
2,"one-to-many, duplicated in human","one-to-many, duplicated in mouse",
3,many-to-many,"one-to-many, duplicated in human",
4,many-to-many,"one-to-many, duplicated in mouse",
5,"one-to-many, duplicated in human, GOC=0","one-to-many, duplicated in human, GOC=25",
6,"one-to-many, duplicated in human, GOC=25","one-to-many, duplicated in human, GOC=50",
7,"one-to-many, duplicated in human, GOC=50","one-to-many, duplicated in human, GOC=75",
8,"one-to-many, duplicated in human, GOC=75","one-to-many, duplicated in human, GOC=100",
9,"one-to-many, duplicated in human, GOC=0","one-to-many, duplicated in human, GOC=100",


In [38]:
pValueDFArr = []
for distMetric in dataProteinOnly.columns[4:9]:
    pValueArr = []
    for rowNum in range(pValueDF.shape[0]):
        translatedGroup1 = list(map(translationTable.get, pValueDF.iloc[rowNum, :].iloc[0].split(", ")))
        translatedGroup2 = list(map(translationTable.get, pValueDF.iloc[rowNum, :].iloc[1].split(", ")))

        if len(translatedGroup1) == 1:
            filteredDF1 = dataProteinOnly[dataProteinOnly["Mouse homology type"] == translatedGroup1[0]][distMetric].dropna()
        elif len(translatedGroup1) == 2:
            filteredDF1 = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedGroup1[0]) & (dataProteinOnly["Duplicated Species"] == translatedGroup1[1])][distMetric].dropna()
        else:
            filteredDF1 = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedGroup1[0]) & (dataProteinOnly["Duplicated Species"] == translatedGroup1[1]) & (dataProteinOnly["Mouse Gene-order conservation score"] == translatedGroup1[2])][distMetric].dropna()
        
        if len(translatedGroup2) == 1:
            filteredDF2 = dataProteinOnly[dataProteinOnly["Mouse homology type"] == translatedGroup2[0]][distMetric].dropna()
        elif len(translatedGroup2) == 2:
            filteredDF2 = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedGroup2[0]) & (dataProteinOnly["Duplicated Species"] == translatedGroup2[1])][distMetric].dropna()
        else:
            filteredDF2 = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedGroup2[0]) & (dataProteinOnly["Duplicated Species"] == translatedGroup2[1]) & (dataProteinOnly["Mouse Gene-order conservation score"] == translatedGroup2[2])][distMetric].dropna()

        pValueArr.append(stats.mannwhitneyu(filteredDF1, filteredDF2)[1])

    pValueDF["P-value"] = pValueArr
    pValueDFArr.append(pValueDF.copy())

In [41]:
emptyCols = pd.DataFrame({
    "": [np.nan] * len(statsDF),
    " ": [np.nan] * len(statsDF)
})


with pd.ExcelWriter("/Users/andrewhsu/Projects/McNair/data/automatedDistances.xlsx") as w:
    for idx, distMetric in enumerate(dataProteinOnly.columns[4:9]):
        finalDF = pd.concat([statsDFArr[idx].astype(str), emptyCols, pValueDFArr[idx].astype(str)], axis=1)
        finalDF.to_excel(w, sheet_name=distMetric, index=False)

In [40]:
statsDFArr

[                                     Category      n     Median         Mean
 0                                  one-to-one  16116  45.779092   255.650467
 1                                 one-to-many   1886   7.789615   461.516873
 2            one-to-many, duplicated in human    480  25.503485  1012.029178
 3     one-to-many, duplicated in human, GOC=0    221  29.224520   393.903635
 4    one-to-many, duplicated in human, GOC=25     46   4.922673    82.496779
 5    one-to-many, duplicated in human, GOC=50     69  17.227953    72.087825
 6    one-to-many, duplicated in human, GOC=75     28  23.071310   171.103285
 7   one-to-many, duplicated in human, GOC=100    116  49.904925  3320.357744
 8            one-to-many, duplicated in mouse   1405   5.020403   273.327189
 9     one-to-many, duplicated in mouse, GOC=0    567   1.765446   142.361515
 10   one-to-many, duplicated in mouse, GOC=25    143   0.118407   692.697554
 11   one-to-many, duplicated in mouse, GOC=50    198   2.765075

# Random Sampling

In [4]:
# TEC
def TEC(humanOrtholog, mouseOrtholog):
    # Turns the vectors binary. So if the expression is greater than 1, we consider that "expressed."
    humanOrthoBinary = (humanOrtholog.iloc[:, :-1] > 1).iloc[0, :]
    mouseOrthoBinary = (mouseOrtholog.iloc[:, :-1] > 1).iloc[0, :]

    humanOnlyTissueNum = (humanOrthoBinary & ~mouseOrthoBinary).sum()
    mouseOnlyTissueNum = (mouseOrthoBinary & ~humanOrthoBinary).sum()

    humanTotalTissue = humanOrthoBinary.sum()
    mouseTotalTissue = mouseOrthoBinary.sum()

    if humanTotalTissue == 0 and mouseTotalTissue == 0:
        return np.nan
    elif humanTotalTissue == 0 or mouseTotalTissue == 0:
        return np.nan
    else:
        return ((humanOnlyTissueNum / humanTotalTissue) + (mouseOnlyTissueNum / mouseTotalTissue)) / 2


In [5]:
columns = ["HumanID", "MouseID", "EuclidDist", "EuclidDistNorm", "EuclidDistLog", "PearDist", "TEC"]
sampleDF = pd.DataFrame(columns=columns)

In [6]:
gtexEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/gtexExpressionProfile.parquet")
emtabEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/emtabExpressionProfile.parquet")

In [7]:
gtexEPProteinOnly = gtexEP[gtexEP["Gene type"] == "protein_coding"]
emtabEPProteinOnly = emtabEP[emtabEP["Gene type"] == "protein_coding"]

In [8]:
# humanGenes = gtexEPProteinOnly.sample(10000, replace=False)
# mouseGenes = emtabEPProteinOnly.sample(10000, replace=False)

In [9]:
# pd.DataFrame({
#     "Human ID": humanGenes.index,
#     "Mouse ID": mouseGenes.index
# }).to_csv("/Users/andrewhsu/Projects/McNair/data/randomGeneIDs.csv", index=False)

In [10]:
geneIDs = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/randomGeneIDs.csv")

In [11]:
humanGenes = gtexEPProteinOnly[gtexEPProteinOnly.index.isin(geneIDs["Human ID"])]
mouseGenes = emtabEPProteinOnly[emtabEPProteinOnly.index.isin(geneIDs["Mouse ID"])]

In [12]:
humanGenes

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSG00000000971,3.772009,18.315853,28.689909,54.564095,10.287676,199.909653,2.872952,16.362883,protein_coding
ENSG00000001036,11.936454,31.576033,22.964733,18.165680,30.105265,24.064280,18.745176,33.031639,protein_coding
ENSG00000001084,10.668489,18.914995,21.850657,4.286869,7.587957,23.261522,6.228732,18.729752,protein_coding
ENSG00000001167,11.078234,10.240786,10.754413,5.289579,7.796001,6.300948,5.891113,10.038259,protein_coding
ENSG00000001561,16.347681,12.644885,11.827224,9.480109,9.981854,4.387220,6.151659,12.146295,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSG00000293663,0.012648,0.013364,0.019085,0.007283,0.003708,0.001775,0.024269,0.020217,protein_coding
ENSG00000293689,0.067410,0.212483,0.139108,0.077048,0.118661,0.117689,0.059908,0.139764,protein_coding
ENSG00000300293,0.003806,0.003848,0.004492,0.001905,0.068468,0.003040,0.000889,0.001678,protein_coding


In [13]:
humanGenesNorm = humanGenes.iloc[:, :-1].div(np.linalg.norm(humanGenes.iloc[:, :-1], axis=1), axis=0)
mouseGenesNorm = mouseGenes.iloc[:, :-1].div(np.linalg.norm(mouseGenes.iloc[:, :-1], axis=1), axis=0)

In [14]:
humanGenesLog = np.log2(humanGenes.iloc[:, :-1] + 1)
mouseGenesLog = np.log2(mouseGenes.iloc[:, :-1] + 1)

In [15]:
euclidDist = np.linalg.norm(humanGenes.iloc[:, :-1] - mouseGenes.iloc[:, :-1].values, axis=1)

In [16]:
euclidDistNorm = np.linalg.norm(humanGenesNorm - mouseGenesNorm.values, axis=1)

In [17]:
euclidDistLog = np.linalg.norm(humanGenesLog - mouseGenesLog.values, axis=1)

In [18]:
humanGenesLog - mouseGenesLog.values

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach
Gene stable ID,,,,,,,,
ENSG00000000971,1.088205,3.263678,4.348386,5.473915,0.956812,-3.201419,1.199724,3.717741
ENSG00000001036,-2.000676,-0.058643,-0.254316,-1.956511,-0.060288,0.534790,2.732160,0.579380
ENSG00000001084,-1.461063,-2.874477,-1.944224,-3.132411,-1.169420,1.772095,0.150754,-1.050820
ENSG00000001167,-5.253714,-6.327388,-5.567020,-7.755599,-6.549396,-6.237816,-3.487423,-5.874005
ENSG00000001561,3.869388,3.187809,1.225114,3.376551,3.444208,2.429541,2.823313,2.699882
...,...,...,...,...,...,...,...,...
ENSG00000293663,0.018133,0.019153,0.027274,0.010469,0.005340,0.002558,0.034594,0.028875
ENSG00000293689,0.094114,0.277965,0.187904,0.107083,0.161773,0.160519,0.083939,0.188735
ENSG00000300293,0.005480,0.005541,0.006466,0.002745,0.095544,0.004380,0.001282,0.002419


In [19]:
pearDist = np.diag(cdist(humanGenes.iloc[:, :-1].values.astype(float), mouseGenes.iloc[:, :-1].values.astype(float), metric="correlation"))

In [20]:
tecValues = [TEC(humanGenes.iloc[rowIdx: rowIdx + 1], mouseGenes.iloc[rowIdx: rowIdx + 1])for rowIdx in range(humanGenes.shape[0])]

In [21]:
sampleDF["HumanID"] = humanGenes.index
sampleDF["MouseID"] = mouseGenes.index
sampleDF["EuclidDist"] = euclidDist
sampleDF["EuclidDistNorm"] = euclidDistNorm
sampleDF["EuclidDistLog"] = euclidDistLog
sampleDF["PearDist"] = pearDist
sampleDF["TEC"] = tecValues

In [22]:
sampleDF.to_csv("/Users/andrewhsu/Projects/McNair/data/randomGenesTable.csv", index=False)
sampleDF.to_parquet("/Users/andrewhsu/Projects/McNair/data/randomGenesTable.parquet", index=False)

# Statistical Analysis for Random Samples

In [11]:
data = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupeBioType.parquet")
dataProteinOnly = data[(data["Human Gene Type"] == "protein_coding") & (data["Mouse Gene Type"] == "protein_coding")]

randomData = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/randomGenesTable.parquet")

In [12]:
randomData

,HumanID,MouseID,EuclidDist,EuclidDistNorm,EuclidDistLog,PearDist,TEC
0,ENSG00000000971,ENSMUSG00000000049,1648.502447,0.322558,9.334532,0.032440,0.250
1,ENSG00000001036,ENSMUSG00000000056,71.067645,0.697079,3.998596,1.349007,0.000
2,ENSG00000001084,ENSMUSG00000000078,152.010822,0.680316,5.442806,0.618386,0.000
3,ENSG00000001167,ENSMUSG00000000088,2119.855406,0.565457,16.948433,1.150569,0.000
4,ENSG00000001561,ENSMUSG00000000094,28.852232,0.976974,8.429828,0.742289,0.375
...,...,...,...,...,...,...,...
9995,ENSG00000293663,ENSMUSG00000109516,0.042077,NaN,0.060123,NaN,NaN
9996,ENSG00000293689,ENSMUSG00000109520,0.355014,NaN,0.476970,NaN,NaN
9997,ENSG00000300293,ENSMUSG00000109528,0.068948,NaN,0.096257,NaN,NaN
9998,ENSG00000300510,ENSMUSG00000109542,1.096681,NaN,1.230796,NaN,NaN


In [13]:
statsDF = pd.DataFrame({
    "Category": ["one-to-one", 
                 "one-to-many", "one-to-many, duplicated in human", "one-to-many, duplicated in human, GOC=0", "one-to-many, duplicated in human, GOC=25", "one-to-many, duplicated in human, GOC=50", "one-to-many, duplicated in human, GOC=75", "one-to-many, duplicated in human, GOC=100",
                 "one-to-many, duplicated in mouse", "one-to-many, duplicated in mouse, GOC=0", "one-to-many, duplicated in mouse, GOC=25", "one-to-many, duplicated in mouse, GOC=50", "one-to-many, duplicated in mouse, GOC=75", "one-to-many, duplicated in mouse, GOC=100",
                 "many-to-many", "random"],
    "n": "",
    "Median": "",
    "Mean": ""
})

pValueDF = pd.DataFrame({
    "Group 1": ["one-to-one", "one-to-one",
                "one-to-many, duplicated in human",
                "many-to-many", "many-to-many",
                "one-to-many, duplicated in human, GOC=0", "one-to-many, duplicated in human, GOC=25", "one-to-many, duplicated in human, GOC=50", "one-to-many, duplicated in human, GOC=75", "one-to-many, duplicated in human, GOC=0",
                "one-to-many, duplicated in mouse, GOC=0", "one-to-many, duplicated in mouse, GOC=25", "one-to-many, duplicated in mouse, GOC=50", "one-to-many, duplicated in mouse, GOC=75", "one-to-many, duplicated in mouse, GOC=0"],

    "Group 2": ["one-to-many, duplicated in human", "one-to-many, duplicated in mouse",
                "one-to-many, duplicated in mouse", 
                "one-to-many, duplicated in human", "one-to-many, duplicated in mouse", 
                "one-to-many, duplicated in human, GOC=25", "one-to-many, duplicated in human, GOC=50", "one-to-many, duplicated in human, GOC=75", "one-to-many, duplicated in human, GOC=100", "one-to-many, duplicated in human, GOC=100",
                "one-to-many, duplicated in mouse, GOC=25", "one-to-many, duplicated in mouse, GOC=50", "one-to-many, duplicated in mouse, GOC=75", "one-to-many, duplicated in mouse, GOC=100", "one-to-many, duplicated in mouse, GOC=100"],
    "P-value": ""
})

In [14]:
translationTable = {
    "one-to-one": "ortholog_one2one",
    "one-to-many": "ortholog_one2many",
    "duplicated in human": "Human",
    "duplicated in mouse": "Mouse",
    "GOC=0": 0,
    "GOC=25": 25,
    "GOC=50": 50,
    "GOC=75": 75,
    "GOC=100": 100,
    "many-to-many": "ortholog_many2many"
}

In [15]:
statsDFArr = []
for distMetric in dataProteinOnly.columns[4:9]:
    nArr = []
    medianArr = []
    meanArr = []

    for category in statsDF["Category"]:
        if category == "random":
            filteredDF = randomData[distMetric].dropna()
        else:
            categoryParts = category.split(", ")
            translatedParts = list(map(translationTable.get, categoryParts))
            if len(translatedParts) == 1:
                filteredDF = dataProteinOnly[dataProteinOnly["Mouse homology type"] == translatedParts[0]][distMetric].dropna()
            elif len(translatedParts) == 2:
                filteredDF = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedParts[0]) & (dataProteinOnly["Duplicated Species"] == translatedParts[1])][distMetric].dropna()
            else:
                filteredDF = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedParts[0]) & (dataProteinOnly["Duplicated Species"] == translatedParts[1]) & (dataProteinOnly["Mouse Gene-order conservation score"] == translatedParts[2])][distMetric].dropna()

        nArr.append(filteredDF.shape[0])
        medianArr.append(filteredDF.median())
        meanArr.append(filteredDF.mean())

    statsDF["n"] = nArr
    statsDF["Median"] = medianArr
    statsDF["Mean"] = meanArr
    statsDFArr.append(statsDF.copy())

In [16]:
pValueDFArr = []
for distMetric in dataProteinOnly.columns[4:9]:
    pValueArr = []
    for rowNum in range(pValueDF.shape[0]):
        translatedGroup1 = list(map(translationTable.get, pValueDF.iloc[rowNum, :].iloc[0].split(", ")))
        translatedGroup2 = list(map(translationTable.get, pValueDF.iloc[rowNum, :].iloc[1].split(", ")))

        if len(translatedGroup1) == 1:
            filteredDF1 = dataProteinOnly[dataProteinOnly["Mouse homology type"] == translatedGroup1[0]][distMetric].dropna()
        elif len(translatedGroup1) == 2:
            filteredDF1 = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedGroup1[0]) & (dataProteinOnly["Duplicated Species"] == translatedGroup1[1])][distMetric].dropna()
        else:
            filteredDF1 = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedGroup1[0]) & (dataProteinOnly["Duplicated Species"] == translatedGroup1[1]) & (dataProteinOnly["Mouse Gene-order conservation score"] == translatedGroup1[2])][distMetric].dropna()
        
        if len(translatedGroup2) == 1:
            filteredDF2 = dataProteinOnly[dataProteinOnly["Mouse homology type"] == translatedGroup2[0]][distMetric].dropna()
        elif len(translatedGroup2) == 2:
            filteredDF2 = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedGroup2[0]) & (dataProteinOnly["Duplicated Species"] == translatedGroup2[1])][distMetric].dropna()
        else:
            filteredDF2 = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == translatedGroup2[0]) & (dataProteinOnly["Duplicated Species"] == translatedGroup2[1]) & (dataProteinOnly["Mouse Gene-order conservation score"] == translatedGroup2[2])][distMetric].dropna()

        pValueArr.append(stats.mannwhitneyu(filteredDF1, filteredDF2)[1])

    pValueDF["P-value"] = pValueArr
    pValueDFArr.append(pValueDF.copy())

In [17]:
emptyCols = pd.DataFrame({
    "": [np.nan] * len(statsDF),
    " ": [np.nan] * len(statsDF)
})


with pd.ExcelWriter("/Users/andrewhsu/Projects/McNair/data/automatedDistancesWithRandom.xlsx") as w:
    for idx, distMetric in enumerate(dataProteinOnly.columns[4:9]):
        finalDF = pd.concat([statsDFArr[idx].astype(str), emptyCols, pValueDFArr[idx].astype(str)], axis=1)
        finalDF.to_excel(w, sheet_name=distMetric, index=False)

# Parental-Daughter Identification

In [3]:
humanColumns = ["Human Gene", "Mouse Parent", "Mouse Daughter", "GOC Parent", "GOC Daughter", "EuclidDist Parent", "EuclidDist Daughter", "EuclidDistNorm Parent", "EuclidDistNorm Daughter", "EuclidDistLog Parent", "EuclidDistLog Daughter", "PearDist Parent", "PearDist Daughter", "TEC Parent", "TEC Daughter"]
mouseColumns = ["Mouse Gene", "Human Parent", "Human Daughter", "GOC Parent", "GOC Daughter", "EuclidDist Parent", "EuclidDist Daughter", "EuclidDistNorm Parent", "EuclidDistNorm Daughter", "EuclidDistLog Parent", "EuclidDistLog Daughter", "PearDist Parent", "PearDist Daughter", "TEC Parent", "TEC Daughter"]
humanParent = pd.DataFrame()
mouseParent = pd.DataFrame()

In [4]:
orthologTable = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupeBioType.parquet")
humanParentTable = orthologTable[(orthologTable["Mouse homology type"] == "ortholog_one2many") & (orthologTable["Duplicated Species"] == "Mouse")]
mouseParentTable = orthologTable[(orthologTable["Mouse homology type"] == "ortholog_one2many") & (orthologTable["Duplicated Species"] == "Human")]

In [5]:
humanParentRows = []
for humanID in humanParentTable["Gene stable ID"].unique():
    humanIDDF = humanParentTable[humanParentTable["Gene stable ID"] == humanID]
    humanMaxGOC = humanIDDF["Mouse Gene-order conservation score"].max()
    humanMinGOC = humanIDDF["Mouse Gene-order conservation score"].min()

    if humanMaxGOC != humanMinGOC:
        parentalCopy = humanIDDF[humanIDDF["Mouse Gene-order conservation score"] == humanMaxGOC]
        daughterCopy = humanIDDF[humanIDDF["Mouse Gene-order conservation score"] == humanMinGOC]

        parentalCopyNAMin = parentalCopy[parentalCopy.isna().sum(axis=1) == parentalCopy.isna().sum(axis=1).min()].sample(n=1)
        daughterCopyNAMin = daughterCopy[daughterCopy.isna().sum(axis=1) == daughterCopy.isna().sum(axis=1).min()].sample(n=1)

        humanParentRow = [humanID, parentalCopyNAMin["Mouse gene stable ID"].item(), daughterCopyNAMin["Mouse gene stable ID"].item(), 
                        parentalCopyNAMin["Mouse Gene-order conservation score"].item(), daughterCopyNAMin["Mouse Gene-order conservation score"].item(),
                        parentalCopyNAMin["EuclidDist"].item(), daughterCopyNAMin["EuclidDist"].item(),
                        parentalCopyNAMin["EuclidDistNorm"].item(), daughterCopyNAMin["EuclidDistNorm"].item(),
                        parentalCopyNAMin["EuclidDistLog"].item(), daughterCopyNAMin["EuclidDistLog"].item(),
                        parentalCopyNAMin["PearDist"].item(), daughterCopyNAMin["PearDist"].item(),
                        parentalCopyNAMin["TEC"].item(), daughterCopyNAMin["TEC"].item()]
        humanParentRows.append(humanParentRow)
humanParent = pd.DataFrame(humanParentRows)
humanParent.columns = humanColumns

In [6]:
humanParent.to_csv("/Users/andrewhsu/Projects/McNair/data/mouseParentDaughter.csv", index=False)
humanParent.to_parquet("/Users/andrewhsu/Projects/McNair/data/mouseParentDaughter.parquet", index=False)

In [7]:
mouseParentRows = []
for mouseID in mouseParentTable["Mouse gene stable ID"].unique():
    mouseIDDF = mouseParentTable[mouseParentTable["Mouse gene stable ID"] == mouseID]
    mouseMaxGOC = mouseIDDF["Mouse Gene-order conservation score"].max()
    mouseMinGOC = mouseIDDF["Mouse Gene-order conservation score"].min()

    if mouseMaxGOC != mouseMinGOC:
        parentalCopy = mouseIDDF[mouseIDDF["Mouse Gene-order conservation score"] == mouseMaxGOC]
        daughterCopy = mouseIDDF[mouseIDDF["Mouse Gene-order conservation score"] == mouseMinGOC]

        parentalCopyNAMin = parentalCopy[parentalCopy.isna().sum(axis=1) == parentalCopy.isna().sum(axis=1).min()].sample(n=1)
        daughterCopyNAMin = daughterCopy[daughterCopy.isna().sum(axis=1) == daughterCopy.isna().sum(axis=1).min()].sample(n=1)

        mouseParentRow = [mouseID, parentalCopyNAMin["Gene stable ID"].item(), daughterCopyNAMin["Gene stable ID"].item(), 
                        parentalCopyNAMin["Mouse Gene-order conservation score"].item(), daughterCopyNAMin["Mouse Gene-order conservation score"].item(),
                        parentalCopyNAMin["EuclidDist"].item(), daughterCopyNAMin["EuclidDist"].item(),
                        parentalCopyNAMin["EuclidDistNorm"].item(), daughterCopyNAMin["EuclidDistNorm"].item(),
                        parentalCopyNAMin["EuclidDistLog"].item(), daughterCopyNAMin["EuclidDistLog"].item(),
                        parentalCopyNAMin["PearDist"].item(), daughterCopyNAMin["PearDist"].item(),
                        parentalCopyNAMin["TEC"].item(), daughterCopyNAMin["TEC"].item()]
        mouseParentRows.append(mouseParentRow)
mouseParent = pd.DataFrame(mouseParentRows)
mouseParent.columns = mouseColumns

In [8]:
mouseParent.to_csv("/Users/andrewhsu/Projects/McNair/data/humanParentDaughter.csv", index=False)
mouseParent.to_parquet("/Users/andrewhsu/Projects/McNair/data/humanParentDaughter.parquet", index=False)

In [9]:
mouseParentTable.sort_values("Mouse gene stable ID")

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score,EuclidDist,EuclidDistNorm,EuclidDistLog,PearDist,TEC,Num Human Dupes,Num Mouse Dupes,Duplicated Species,Human Gene Type,Mouse Gene Type
953,ENSG00000223572,ENSMUSG00000000308,ortholog_one2many,75.0,2569.930198,0.869675,12.033288,0.554442,0.187500,2.0,1,Human,protein_coding,protein_coding
955,ENSG00000237289,ENSMUSG00000000308,ortholog_one2many,100.0,2561.844409,1.182314,11.026921,0.921149,0.187500,2.0,1,Human,protein_coding,protein_coding
1060,ENSG00000129204,ENSMUSG00000000804,ortholog_one2many,0.0,115.484260,0.769929,10.174987,0.869201,0.375000,2.0,1,Human,protein_coding,protein_coding
1062,ENSG00000170832,ENSMUSG00000000804,ortholog_one2many,50.0,94.660307,0.495699,4.109197,0.335662,0.000000,2.0,1,Human,protein_coding,protein_coding
1089,ENSG00000275385,ENSMUSG00000000982,ortholog_one2many,0.0,10.376588,1.049001,4.076298,1.147996,0.333333,3.0,1,Human,protein_coding,protein_coding
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3879084,ENSG00000286175,ENSMUSG00000116652,ortholog_one2many,0.0,NaN,NaN,NaN,NaN,NaN,3.0,1,Human,protein_coding,NaN
3879296,ENSG00000146938,ENSMUSG00000121607,ortholog_one2many,25.0,NaN,NaN,NaN,NaN,NaN,2.0,1,Human,protein_coding,NaN
3879298,ENSG00000165246,ENSMUSG00000121607,ortholog_one2many,0.0,NaN,NaN,NaN,NaN,NaN,2.0,1,Human,protein_coding,NaN
3879370,ENSG00000288841,ENSMUSG00000144229,ortholog_one2many,0.0,NaN,NaN,NaN,NaN,NaN,2.0,1,Human,NaN,NaN


In [10]:
humanParent

,Human Gene,Mouse Parent,Mouse Daughter,GOC Parent,GOC Daughter,EuclidDist Parent,EuclidDist Daughter,EuclidDistNorm Parent,EuclidDistNorm Daughter,EuclidDistLog Parent,EuclidDistLog Daughter,PearDist Parent,PearDist Daughter,TEC Parent,TEC Daughter
0,ENSG00000254647,ENSMUSG00000000215,ENSMUSG00000035804,100.0,0.0,3667.846303,5264.286219,0.001498,0.001428,2.445157,4.066262,5.970543e-07,8.228542e-07,0.583333,0.250000
1,ENSG00000069493,ENSMUSG00000030157,ENSMUSG00000000248,50.0,0.0,495.295522,8.910251,0.940933,0.994561,11.994275,3.925919,1.027792e+00,6.630076e-01,0.000000,0.437500
2,ENSG00000157601,ENSMUSG00000023341,ENSMUSG00000000386,75.0,25.0,31.889421,31.041610,0.814844,0.718185,5.317110,6.218484,1.319186e+00,7.603217e-01,0.062500,0.187500
3,ENSG00000146425,ENSMUSG00000095677,ENSMUSG00000000579,50.0,0.0,346.575685,352.216022,0.499185,0.570997,13.120709,15.830996,4.088576e-01,4.344678e-01,0.187500,0.375000
4,ENSG00000089127,ENSMUSG00000052776,ENSMUSG00000001168,100.0,0.0,26.897303,20.208340,0.872478,0.943921,3.505326,6.439363,6.335573e-01,7.247547e-01,0.071429,0.428571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
222,ENSG00000151327,ENSMUSG00000094103,ENSMUSG00000095595,75.0,50.0,34.377397,34.377397,NaN,NaN,9.680407,9.680407,NaN,NaN,NaN,NaN
223,ENSG00000026297,ENSMUSG00000095687,ENSMUSG00000094724,50.0,25.0,11.552547,12.033663,0.652015,0.666927,2.772407,2.871943,8.719934e-01,9.140217e-01,0.062500,0.062500
224,ENSG00000188324,ENSMUSG00000095401,ENSMUSG00000095075,25.0,0.0,0.009028,0.009028,NaN,NaN,0.012357,0.012357,NaN,NaN,NaN,NaN
225,ENSG00000146385,ENSMUSG00000100004,ENSMUSG00000096442,100.0,0.0,0.070029,0.021049,1.314738,NaN,0.097703,0.029916,1.205051e+00,NaN,NaN,NaN


In [11]:
humanStats = {}
for distMetric in humanParent.columns[5::2]:
    numParentalHigher = (humanParent[distMetric] > humanParent[distMetric.replace("Parent", "Daughter")]).sum()
    numDaughterHigher = (humanParent[distMetric] < humanParent[distMetric.replace("Parent", "Daughter")]).sum()
    numEqual = (humanParent[distMetric] == humanParent[distMetric.replace("Parent", "Daughter")]).sum()
    humanStats[distMetric.replace(" Parent", "")] = [numParentalHigher, numDaughterHigher, numEqual]

humanStatsDF = pd.DataFrame(humanStats)
humanStatsDF.index = ["Parental", "Daughter", "Equal"]

In [12]:
humanStatsDF

,EuclidDist,EuclidDistNorm,EuclidDistLog,PearDist,TEC
Parental,73,38,48,53,11
Daughter,98,97,123,82,42
Equal,39,4,39,4,12


In [13]:
mouseStats = {}
for distMetric in mouseParent.columns[5::2]:
    numParentalHigher = (mouseParent[distMetric] > mouseParent[distMetric.replace("Parent", "Daughter")]).sum()
    numDaughterHigher = (mouseParent[distMetric] < mouseParent[distMetric.replace("Parent", "Daughter")]).sum()
    numEqual = (mouseParent[distMetric] == mouseParent[distMetric.replace("Parent", "Daughter")]).sum()
    mouseStats[distMetric.replace(" Parent", "")] = [numParentalHigher, numDaughterHigher, numEqual]

mouseStatsDF = pd.DataFrame(mouseStats)
mouseStatsDF.index = ["Parental", "Daughter", "Equal"]

In [14]:
mouseStatsDF

,EuclidDist,EuclidDistNorm,EuclidDistLog,PearDist,TEC
Parental,47,47,44,47,4
Daughter,73,60,76,60,18
Equal,0,0,0,0,19


In [17]:
with pd.ExcelWriter("/Users/andrewhsu/Projects/McNair/data/parentDaughterStats.xlsx") as w:
    humanStatsDF.to_excel(w, sheet_name="Human", index=False)
    mouseStatsDF.to_excel(w, sheet_name="Mouse", index=False)